**🔑 00 — Tổng quan & Khung phân tích**

Notebook mở đầu: nạp dữ liệu, nêu bài toán, và giải thích **vì sao** các notebook sau
phải kiểm soát nhiễu.

> ⚠️ **Ba nguyên tắc xuyên suốt**
> 1. **Tách riêng Uber (`dfU`) và Lyft (`dfL`)** — 2 hãng có công thức giá khác nhau
>    (đơn giá/dặm khác, danh mục dịch vụ không trùng nhau). Chỉ dùng `df` khi cần so sánh.
> 2. Giá thô bị `quãng đường × loại dịch vụ` chi phối → phải kiểm soát khi xét yếu tố khác.
> 3. Uber không có dữ liệu surge → phân tích hệ số nhân chỉ dùng **Lyft**.

**0. Nạp dữ liệu**

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import importlib, _common; importlib.reload(_common)   # nap lai neu _common.py vua sua
from _common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt

setup()
df, dfU, dfL = load()

**1. Kiểm tra nhanh**

In [ ]:
print("="*64)
print(f"So chuyen        : {len(df):,}")
print(f"Khoang thoi gian : {df.date_local.min().date()} -> {df.date_local.max().date()}"
      f"  ({df.date_local.nunique()} ngay)")
print(f"Hang / dich vu   : {df.cab_type.nunique()} / {df.name.nunique()}")
print(f"Khu vuc / tuyen  : {df.source.nunique()} / {df.groupby(['source','destination']).ngroups}")
print("="*64)
print(f"Gia         : {df.price.min():>6.2f} - {df.price.max():>6.2f} USD  (TB {df.price.mean():.2f})")
print(f"Quang duong : {df.distance.min():>6.2f} - {df.distance.max():>6.2f} dam (TB {df.distance.mean():.2f})")
print(f"He so nhan  : {sorted(df.surge_multiplier.unique())}")

**2. ⚠️ Vì sao không so sánh giá thô trực tiếp được**

Đây là cạm bẫy lớn nhất của bộ dữ liệu này.

In [ ]:
cmp = df.groupby("source").agg(
        so_chuyen=("price","size"), gia_TB=("price","mean"),
        quang_duong_TB=("distance","mean"),
        ty_le_xe_sang=("name", lambda s: s.isin(
            ["Lux","Lux Black","Lux Black XL","Black","Black SUV"]).mean()*100),
      ).sort_values("gia_TB", ascending=False)
display(cmp.round(2))
r1=np.corrcoef(cmp.gia_TB,cmp.quang_duong_TB)[0,1]
r2=np.corrcoef(cmp.gia_TB,cmp.ty_le_xe_sang)[0,1]
print(f"r (gia TB cua khu vs quang duong TB) = {r1:.3f}   <- gan nhu 1!")
print(f"r (gia TB cua khu vs ty le xe sang)  = {r2:.3f}")
print()
print("=> Chenh lech gia giua cac khu gan nhu HOAN TOAN do do dai chuyen di.")
print("   Bao cao 'Boston University dat nhat' la KET LUAN SAI.")

**3. ⚠️ Vì sao số chuyến KHÔNG phải nhu cầu**

In [ ]:
h=dfL.groupby(["source","hour_local"]).size()
g=dfL.groupby("hour_local").agg(n=("price","size"), s=("is_surge","mean"))
print(f"So bao gia moi (khu x gio): CV = {h.std()/h.mean():.3f}   <- qua deu")
print(f"r (so bao gia trong gio) vs (ty le surge) = {np.corrcoef(g.n,g.s)[0,1]:.3f}")
print()
print("=> So chuyen la LICH THU THAP cua nguoi crawl, khong phai nhu cau that.")
print("   Muon do ap luc cau, dung TY LE SURGE (chi co o Lyft).")

**4. ⚠️ Vì sao phải TÁCH RIÊNG Uber và Lyft**

Hai hãng dùng **công thức giá khác nhau** — không được gộp chung khi phân tích giá.

In [ ]:
print("=== QUY MO ===")
for n, d in [("Uber", dfU), ("Lyft", dfL)]:
    print(f"  {n}: {len(d):,} chuyen | {d.name.nunique()} dich vu | "
          f"gia TB {d.price.mean():.2f} USD | q.duong TB {d.distance.mean():.2f} dam")

print("\n=== DANH MUC DICH VU (khong trung nhau chut nao) ===")
print(f"  Uber: {sorted(dfU.name.unique())}")
print(f"  Lyft: {sorted(dfL.name.unique())}")

print("\n=== GIA theo DAI QUANG DUONG — khoang cach GIAN RA ===")
df["dai_qd"] = pd.cut(df.distance, [0,1,2,3,4,10], labels=["<1","1-2","2-3","3-4",">4"])
display(so_sanh_hang(df, "dai_qd"))
print("  -> Uber cang re hon Lyft khi di cang xa => DON GIA/DAM khac nhau,")
print("     khong phai chi lech mot hang so.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
p = df.pivot_table(index="dai_qd", columns="cab_type", values="price", aggfunc="mean")
x = np.arange(len(p))
ax[0].bar(x-0.2, p["Uber"], width=.38, color=MAU_HANG["Uber"], label="Uber", zorder=3)
ax[0].bar(x+0.2, p["Lyft"], width=.38, color=MAU_HANG["Lyft"], label="Lyft", zorder=3)
ax[0].set_xticks(x); ax[0].set_xticklabels(p.index)
ax[0].set_xlabel("Dai quang duong (dam)"); ax[0].set_ylabel("Gia TB (USD)")
ax[0].set_title("Gia theo quang duong", fontweight="bold"); ax[0].legend(frameon=False)

xs = np.linspace(0, 8, 50)
for h, d in [("Uber", dfU), ("Lyft", dfL)]:
    z = np.polyfit(d.distance, d.price, 1)
    ax[1].plot(xs, np.polyval(z, xs), color=MAU_HANG[h], lw=2.5,
               label=f"{h}: {z[1]:.2f} + {z[0]:.2f}/dam")
ax[1].set_xlabel("Quang duong (dam)"); ax[1].set_ylabel("Gia (USD)")
ax[1].set_title("Duong gia 2 hang", fontweight="bold"); ax[1].legend(frameon=False)
for a in ax: a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

---
**📁 Các notebook phân tích**

| File | Nội dung |
|---|---|
| `01_location.ipynb` | 📍 Khu vực → giá & hệ số nhân (12 chân dung khu) |
| `02_time.ipynb` | 🕐 Giờ, thứ, cuối tuần → giá & hệ số nhân |
| `03_weather.ipynb` | 🌦️ Thời tiết → giá & hệ số nhân |
| `04_distance_baseprice.ipynb` | 📏 Quãng đường → giá cơ sở (giải mã công thức) |
| `05_tong_hop.ipynb` | ⚖️ So sức mạnh 4 nhóm + hồi quy kiểm soát |

Tất cả dùng chung `_common.py` (nạp dữ liệu, bảng màu, hàm `eta`, `base_price_model`).